In [1]:
#X1,Y1,X2,Y2,a,b = PolynomialRing(ZZ,'X1,Y1,X2,Y2,a,b').gens()
X1,Y1,X2,Y2,b = PolynomialRing(ZZ,'X1,Y1,X2,Y2,b').gens()
a = 0

In [2]:
from utils import GLVCurve, Registers
from dcp import prepare_simpleVpolynomial, prepare_Vpolynomial

def remove_y(polynomial):
    new_polynomial = polynomial
    for X,Y in [(X1,Y1),(X2,Y2)]:
        for d in range(2,new_polynomial.degree(Y)+1):
            coef = new_polynomial.coefficient(Y**d)
            new_polynomial-=coef*Y**d
            new_polynomial+=coef*(X**3+X*a+b)**(d//2)*Y**(d%2)
        l = new_polynomial.coefficient(Y)
        ab = new_polynomial-Y*l
        new_polynomial = l**2*(X**3+X*a+b)-ab**2
    factors = new_polynomial.factor()
    return factors

def dcp_statistics(polynomial, curve, original_poly):
    field = curve.base_field()
    x,x1,x2,n,d = PolynomialRing(field,'x,x1,x2,n,d').fraction_field().gens()
    if a!=0:
        V = x.parent()(polynomial(x1,1,x2,1,curve.a4(),curve.a6()))
    else:
        V = x.parent()(polynomial(x1,1,x2,1,curve.a6()))
    V = V(x,x,n/d,n,d).numerator()
    V = PolynomialRing(field,'x,n,d')(V)
    counter = 0
    counter_curve = 0
    counter_original_poly = 0
    for k in range(2,20):
        kmap = curve.multiplication_by_m(k)
        num,den = kmap[0].numerator()(x,1), kmap[0].denominator()(x,1) 
        f = V(x,num,den).numerator().univariate_polynomial()
        roots = f.roots()
        roots = [x for x,e in roots if x!=0]
        counter+=(len(roots)>=1)
        for r in roots:
            try:
                P = curve.lift_x(r)
                Q = k*P
                if a!=0:
                    assert polynomial(P[0],P[1],Q[0],Q[1],curve.a4(),curve.a6())==0
                else:
                    assert polynomial(P[0],P[1],Q[0],Q[1],curve.a6())==0
                counter_curve+=1
                break
            except:
                continue
        for r in roots:
            try:
                P = curve.lift_x(r)
                Q = k*P
                if a!=0:
                    assert original_poly(P[0],P[1],Q[0],Q[1],curve.a4(),curve.a6())==0
                else:
                    assert original_poly(P[0],P[1],Q[0],Q[1],curve.a6())==0
                counter_original_poly+=1
                break
            except:
                continue
    return counter_curve, counter_original_poly

def random_curve(bits):
    while True:
        p = random_prime(2**bits-1,True,2**(bits-1))
        F = GF(p)
        j = F.random_element()
        E = EllipticCurve_from_j(j)
        n = E.order()
        if n.is_prime():
            break
    return E

def random_a0_curve(bits):
    while True:
        p = random_prime(2**bits-1,True,2**(bits-1))
        F = GF(p)
        b = F.random_element()
        try:
            E = EllipticCurve(F,[0,b])
        except:
            continue
        n = E.order()
        if n.is_prime():
            break
    return E   

def is_sign_agnostic(polynomial):
    result = 0
    if polynomial(Y1=-Y1)==polynomial or polynomial(Y1=-Y1)==-polynomial:
        result+=1
    if polynomial(Y2=-Y2)==polynomial or polynomial(Y2=-Y2)==-polynomial:
        result+=1
    if result>0:
        return result
    if polynomial(Y1=-Y1,Y2=-Y2)==polynomial or polynomial(Y1=-Y1,Y2=-Y2)==-polynomial:
        return 1
    return 0

def is_symmetric(polynomial):
    return polynomial(X1=X2,X2=X1,Y1=Y2,Y2=Y1)==polynomial or polynomial(X1=X2,X2=X1,Y1=Y2,Y2=Y1)==-polynomial

def polynomial_string(polynomial):
    return str(polynomial).replace("^","**")


def Vdegrees(polynomial):
    glv = GLVCurve()
    curve = random_a0_curve(20)
    glv.curve = curve 
    glv.beta = 1 # does not change the degree
    registers = Registers()
    polynomial = registers.ring(polynomial)
    simpleV = prepare_simpleVpolynomial(polynomial, glv, registers)
    V = prepare_Vpolynomial(polynomial, registers, glv)
    return simpleV.degree(), len(simpleV.monomials()), (V.degrees(),V.degree()), len(V.monomials())


In [3]:
pols = {X2 - X1 + 1,
 X2 - X1 + 2,
 X2 + X1,
 Y2*Y1 + 1,
 Y2*Y1 - 3*b,
 Y2*Y1 + 3*b,
 X2*X1 + 1,
 X2*X1 + Y2*Y1,
 Y2*X1 + X2*Y1,
 X2^2*X1 - 2*X2*X1^2 + X1^3 - Y2^2 + 2*Y2*Y1 - Y1^2,
 2*X2^2*X1 - 4*X2*X1^2 + 2*X1^3 - Y2^2 + 2*Y2*Y1 - Y1^2,
 X2^3 - 3*X2*X1^2 + 2*X1^3 - Y2^2 + 2*Y2*Y1 - Y1^2,
 X2^3 - 3*X2^2*X1 + 3*X2*X1^2 - X1^3 - Y2^2 + 2*Y2*Y1 - Y1^2,
 2*X2^4 + 4*X2^3*X1 + 6*X2^2*X1^2 + 4*X2*X1^3 + 2*X1^4 - 3*Y2^2*X2 - 3*Y2^2*X1 - 6*Y2*X2*Y1 - 6*Y2*X1*Y1 - 3*X2*Y1^2 - 3*X1*Y1^2}

In [4]:
all_factors = []
for f in pols:
    all_factors.extend([(f,r) for r,e in remove_y(f)])
all_factors

[(X1 + X2, X1 + X2),
 (Y1*X2 + X1*Y2, -X1 + X2),
 (Y1*X2 + X1*Y2, -X1^2*X2^2 + X1*b + X2*b),
 (2*X1^3 - 3*X1^2*X2 + X2^3 - Y1^2 + 2*Y1*Y2 - Y2^2, -X1 + X2),
 (2*X1^3 - 3*X1^2*X2 + X2^3 - Y1^2 + 2*Y1*Y2 - Y2^2,
  -X1^4 + 4*X1^3*X2 + 8*X1*b + 4*X2*b),
 (X1*X2 + 1, X1*X2 + 1),
 (-X1 + X2 + 1, -X1 + X2 + 1),
 (-X1 + X2 + 2, -X1 + X2 + 2),
 (Y1*Y2 - 3*b, -X1^3*X2^3 - X1^3*b - X2^3*b + 8*b^2),
 (Y1*Y2 + 1, X1^3*X2^3 + X1^3*b + X2^3*b + b^2 - 1),
 (2*X1^4 + 4*X1^3*X2 + 6*X1^2*X2^2 + 4*X1*X2^3 + 2*X2^4 - 3*X1*Y1^2 - 3*Y1^2*X2 - 6*X1*Y1*Y2 - 6*Y1*X2*Y2 - 3*X1*Y2^2 - 3*X2*Y2^2,
  X1^2 + X1*X2 + X2^2),
 (2*X1^4 + 4*X1^3*X2 + 6*X1^2*X2^2 + 4*X1*X2^3 + 2*X2^4 - 3*X1*Y1^2 - 3*Y1^2*X2 - 6*X1*Y1*Y2 - 6*Y1*X2*Y2 - 3*X1*Y2^2 - 3*X2*Y2^2,
  -X1^4 + 4*X1^3*X2 + 6*X1^2*X2^2 + 4*X1*X2^3 - X2^4 + 24*X1*b + 24*X2*b),
 (2*X1^3 - 4*X1^2*X2 + 2*X1*X2^2 - Y1^2 + 2*Y1*Y2 - Y2^2, -X1 + X2),
 (2*X1^3 - 4*X1^2*X2 + 2*X1*X2^2 - Y1^2 + 2*Y1*Y2 - Y2^2,
  -X1^4 + 6*X1^3*X2 - 7*X1^2*X2^2 + 2*X1*X2^3 - X2^4 + 8*X1*b),
 (Y1

In [5]:
bits = 16
curve = random_a0_curve(bits)

In [6]:
for poly in set(xpoly for _,xpoly in all_factors):
    original_poly = filter(lambda x: x[1]==poly, all_factors).__next__()[0]
    print(poly,";", original_poly,dcp_statistics(poly,curve,original_poly))

-X1^3*X2^3 - X1^3*b - X2^3*b + 8*b^2 ; Y1*Y2 - 3*b (5, 4)
X1 + X2 ; X1 + X2 (2, 2)
-X1^4 + 4*X1^3*X2 + 6*X1^2*X2^2 + 4*X1*X2^3 - X2^4 + 24*X1*b + 24*X2*b ; 2*X1^4 + 4*X1^3*X2 + 6*X1^2*X2^2 + 4*X1*X2^3 + 2*X2^4 - 3*X1*Y1^2 - 3*Y1^2*X2 - 6*X1*Y1*Y2 - 6*Y1*X2*Y2 - 3*X1*Y2^2 - 3*X2*Y2^2 (5, 1)
-X1^2*X2^2 + X1*b + X2*b ; Y1*X2 + X1*Y2 (8, 5)
-X1^4 + 6*X1^3*X2 - 7*X1^2*X2^2 + 2*X1*X2^3 - X2^4 + 8*X1*b ; 2*X1^3 - 4*X1^2*X2 + 2*X1*X2^2 - Y1^2 + 2*Y1*Y2 - Y2^2 (6, 3)
X1*X2 + 1 ; X1*X2 + 1 (4, 4)
-X1 + X2 ; Y1*X2 + X1*Y2 (0, 0)
-X1 + X2 + 1 ; -X1 + X2 + 1 (6, 6)
-X1 + X2 + 2 ; -X1 + X2 + 2 (8, 8)
X1^2 + X1*X2 + X2^2 ; 2*X1^4 + 4*X1^3*X2 + 6*X1^2*X2^2 + 4*X1*X2^3 + 2*X2^4 - 3*X1*Y1^2 - 3*Y1^2*X2 - 6*X1*Y1*Y2 - 6*Y1*X2*Y2 - 3*X1*Y2^2 - 3*X2*Y2^2 (0, 0)
X1^3*X2^3 - X1^2*X2^2 + X1^3*b + X2^3*b + b^2 ; X1*X2 + Y1*Y2 (13, 8)
-4*X1^2*X2^2 - X2^4 + 4*X1*b ; X1^3 - 2*X1^2*X2 + X1*X2^2 - Y1^2 + 2*Y1*Y2 - Y2^2 (2, 1)
-X1^4 + 4*X1^3*X2 + 8*X1*b + 4*X2*b ; 2*X1^3 - 3*X1^2*X2 + X2^3 - Y1^2 + 2*Y1*Y2 - Y2^2 (0

In [7]:
to_remove = [-X1^4 + 4*X1^3*X2 + 8*X1*b + 4*X2*b, X1^2 + X1*X2 + X2^2, -X1+X2]
for f,r in all_factors:
    if not r in to_remove:
        print(f"polynomial = {polynomial_string(f)}")
        print(f"xpoly = {polynomial_string(r)}")
        print(f"Y-sign symmetry = {is_sign_agnostic(f)}")
        print(f"variable symmetry = {is_symmetric(f)}")
        print(f"degree = {f.degree()}, xdegree = {r.degree()}")
        print("V stats: simpleV deg = {0}, simpleV monoms = {1}, V deg = {2}, V monoms = {3}".format(*Vdegrees(r)))
        print("-"*10)


polynomial = X1 + X2
xpoly = X1 + X2
Y-sign symmetry = 2
variable symmetry = True
degree = 1, xdegree = 1
V stats: simpleV deg = 2, simpleV monoms = 2, V deg = ((2, 2, 2, 2, 2), 6), V monoms = 9
----------
polynomial = Y1*X2 + X1*Y2
xpoly = -X1**2*X2**2 + X1*b + X2*b
Y-sign symmetry = 1
variable symmetry = True
degree = 2, xdegree = 4
V stats: simpleV deg = 4, simpleV monoms = 1, V deg = ((4, 4, 4, 4, 4), 12), V monoms = 6
----------
polynomial = X1*X2 + 1
xpoly = X1*X2 + 1
Y-sign symmetry = 2
variable symmetry = True
degree = 2, xdegree = 2
V stats: simpleV deg = 2, simpleV monoms = 2, V deg = ((2, 2, 2, 2, 2), 6), V monoms = 9
----------
polynomial = -X1 + X2 + 1
xpoly = -X1 + X2 + 1
Y-sign symmetry = 2
variable symmetry = False
degree = 1, xdegree = 1
V stats: simpleV deg = 2, simpleV monoms = 3, V deg = ((2, 2, 2, 2, 2), 6), V monoms = 18
----------
polynomial = -X1 + X2 + 2
xpoly = -X1 + X2 + 2
Y-sign symmetry = 2
variable symmetry = False
degree = 1, xdegree = 1
V stats: simpleV 

In [8]:
"""
Output, with comments
root ration = ration of the number of roots of polynomial and xpolynomial (given by nonequivalent transformation) for secp256k1

polynomial = X1 + X2
xpoly = X1 + X2
Y-sign symmetry = 2
variable symmetry = True
degree = 1, xdegree = 1
V stats: simpleV deg = 2, simpleV monoms = 2, V deg = 6, V monoms = 9
#comment: root ratio: 1/1
----------
polynomial = 2*X1**3 - 4*X1**2*X2 + 2*X1*X2**2 - Y1**2 + 2*Y1*Y2 - Y2**2
xpoly = -X1**4 + 6*X1**3*X2 - 7*X1**2*X2**2 + 2*X1*X2**3 - X2**4 + 8*X1*b
Y-sign symmetry = 1
variable symmetry = False
degree = 3, xdegree = 4
V stats: simpleV deg = 8, simpleV monoms = 5, V deg = 24, V monoms = 165
#comment: root ratio: 1/2
----------
polynomial = -X1**3 + 3*X1**2*X2 - 3*X1*X2**2 + X2**3 - Y1**2 + 2*Y1*Y2 - Y2**2
xpoly = -4*X1**4 + 4*X1**3*X2 - 9*X1**2*X2**2 - 4*X1*b + 4*X2*b
Y-sign symmetry = 1
variable symmetry = False
degree = 3, xdegree = 4
V stats: simpleV deg = 6, simpleV monoms = 3, V deg = 16, V monoms = 33
#comment: root ratio: 1/2
----------
polynomial = X1*X2 + Y1*Y2
xpoly = X1**3*X2**3 - X1**2*X2**2 + X1**3*b + X2**3*b + b**2
Y-sign symmetry = 1
variable symmetry = True
degree = 2, xdegree = 6
V stats: simpleV deg = 6, simpleV monoms = 2, V deg = 18, V monoms = 32
#comment: root ratio: 1/2
----------
polynomial = Y1*X2 + X1*Y2
xpoly = -X1**2*X2**2 + X1*b + X2*b
Y-sign symmetry = 1
variable symmetry = True
degree = 2, xdegree = 4
V stats: simpleV deg = 4, simpleV monoms = 1, V deg = 12, V monoms = 6
#comment: root ratio: 1/2
----------
polynomial = -X1 + X2 + 1
xpoly = -X1 + X2 + 1
Y-sign symmetry = 2
variable symmetry = False
degree = 1, xdegree = 1
V stats: simpleV deg = 2, simpleV monoms = 3, V deg = 6, V monoms = 18
#comment: root ratio: 1/1
----------
polynomial = -X1 + X2 + 2
xpoly = -X1 + X2 + 2
Y-sign symmetry = 2
variable symmetry = False
degree = 1, xdegree = 1
V stats: simpleV deg = 2, simpleV monoms = 3, V deg = 6, V monoms = 18
#comment: root ratio: 1/1
----------
polynomial = X1*X2 + 1
xpoly = X1*X2 + 1
Y-sign symmetry = 2
variable symmetry = True
degree = 2, xdegree = 2
V stats: simpleV deg = 2, simpleV monoms = 2, V deg = 6, V monoms = 9
#comment: root ratio: 1/1
----------
polynomial = X1**3 - 2*X1**2*X2 + X1*X2**2 - Y1**2 + 2*Y1*Y2 - Y2**2
xpoly = -4*X1**2*X2**2 - X2**4 + 4*X1*b
Y-sign symmetry = 1
variable symmetry = False
degree = 3, xdegree = 4
V stats: simpleV deg = 6, simpleV monoms = 2, V deg = 20, V monoms = 54
#comment: root ratio: 1/2
----------
polynomial = 2*X1**4 + 4*X1**3*X2 + 6*X1**2*X2**2 + 4*X1*X2**3 + 2*X2**4 - 3*X1*Y1**2 - 3*Y1**2*X2 - 6*X1*Y1*Y2 - 6*Y1*X2*Y2 - 3*X1*Y2**2 - 3*X2*Y2**2
xpoly = -X1**4 + 4*X1**3*X2 + 6*X1**2*X2**2 + 4*X1*X2**3 - X2**4 + 24*X1*b + 24*X2*b
Y-sign symmetry = 1
variable symmetry = True
degree = 4, xdegree = 4
V stats: simpleV deg = 8, simpleV monoms = 5, V deg = 24, V monoms = 165
#comment: root ratio: 1/2
----------
polynomial = Y1*Y2 + 3*b
xpoly = -X1**3*X2**3 - X1**3*b - X2**3*b + 8*b**2
Y-sign symmetry = 1
variable symmetry = True
degree = 2, xdegree = 6
V stats: simpleV deg = 6, simpleV monoms = 1, V deg = 18, V monoms = 10
#comment: root ratio: 1/2
----------
polynomial = Y1*Y2 - 3*b
xpoly = -X1**3*X2**3 - X1**3*b - X2**3*b + 8*b**2
Y-sign symmetry = 1
variable symmetry = True
degree = 2, xdegree = 6
V stats: simpleV deg = 6, simpleV monoms = 1, V deg = 18, V monoms = 10
#comment: root ratio: 1/2
----------
polynomial = Y1*Y2 + 1
xpoly = X1**3*X2**3 + X1**3*b + X2**3*b + b**2 - 1
Y-sign symmetry = 1
variable symmetry = True
degree = 2, xdegree = 6
V stats: simpleV deg = 6, simpleV monoms = 2, V deg = 18, V monoms = 31
#comment: root ratio: 1/2
----------



"""

'\nOutput, with comments\nroot ration = ration of the number of roots of polynomial and xpolynomial (given by nonequivalent transformation) for secp256k1\n\npolynomial = X1 + X2\nxpoly = X1 + X2\nY-sign symmetry = 2\nvariable symmetry = True\ndegree = 1, xdegree = 1\nV stats: simpleV deg = 2, simpleV monoms = 2, V deg = 6, V monoms = 9\n#comment: root ratio: 1/1\n----------\npolynomial = 2*X1**3 - 4*X1**2*X2 + 2*X1*X2**2 - Y1**2 + 2*Y1*Y2 - Y2**2\nxpoly = -X1**4 + 6*X1**3*X2 - 7*X1**2*X2**2 + 2*X1*X2**3 - X2**4 + 8*X1*b\nY-sign symmetry = 1\nvariable symmetry = False\ndegree = 3, xdegree = 4\nV stats: simpleV deg = 8, simpleV monoms = 5, V deg = 24, V monoms = 165\n#comment: root ratio: 1/2\n----------\npolynomial = -X1**3 + 3*X1**2*X2 - 3*X1*X2**2 + X2**3 - Y1**2 + 2*Y1*Y2 - Y2**2\nxpoly = -4*X1**4 + 4*X1**3*X2 - 9*X1**2*X2**2 - 4*X1*b + 4*X2*b\nY-sign symmetry = 1\nvariable symmetry = False\ndegree = 3, xdegree = 4\nV stats: simpleV deg = 6, simpleV monoms = 3, V deg = 16, V monoms =

In [10]:
"""jacobian-0:madd, jacobian-0:add-1998-cmo, jacobian-0:add-2007-bl, jacobian-0:add-1998-cmo-2, jacobian-0:mmadd-2007-bl, jacobian-0:madd-2007-bl, jacobian-0:madd-2008-g, projective:add-1998-cmo, projective:add-1998-cmo-2, projective:madd-1998-cmo, projective:mmadd-1998-cmo, jacobian:madd, jacobian:add-1998-cmo, jacobian:add-2007-bl, jacobian:add-1998-cmo-2, jacobian:mmadd-2007-bl, jacobian:madd-2007-bl, jacobian:madd-2008-g, xyzz:add-2008-s, xyzz:madd-2008-s, xyzz:mmadd-2008-s, modified:madd-2009-bl, modified:add-1998-cmo-2, modified:add-2009-bl, (2): 
{f4} 

jacobian-0:zadd-2007-m, jacobian:zadd-2007-m, (2): 
{f5} 

jacobian-0:madd-2004-hmv, jacobian:madd-2004-hmv, (2): 
{f2} 

jacobian-0:add-1986-cc, jacobian-0:add-2001-b, jacobian:add-1986-cc, jacobian:add-2001-b, (2): 
{f1} 

projective:add-2002-bj, projective:add-2007-bl, (2): 
{f1, f3} 

modified:mmadd-2009-bl, (4): 
{f4, f6, f7} 


projective:madd-2015-rcb (5):
{f1, f8, f9, f10, f11}


f1 = X1 + X2
f2 = 2*X1^3 - 4*X1^2*X2 + 2*X1*X2^2 - Y1^2 + 2*Y1*Y2 - Y2^2 = 2*X1*(X1-X2)^2 - (Y1-Y2)^2 
f3 = 2*X1^4 + 4*X1^3*X2 + 6*X1^2*X2^2 + 4*X1*X2^3 + 2*X2^4 - 3*X1*Y1^2 - 3*X2*Y1^2 - 6*X1*Y1*Y2 - 6*X2*Y1*Y2 - 3*X1*Y2^2 - 3*X2*Y2^2 = X1^4+X2^4+(X1+X2)^4- 3*(X1+X2)*(Y1+Y2)^2 
f4 = X1^3 - 3*X1^2*X2 + 3*X1*X2^2 - X2^3 + Y1^2 - 2*Y1*Y2 + Y2^2 = (X1-X2)^3 + (Y1-Y2)^2
f5 = X1^3 - 2*X1^2*X2 + X1*X2^2 - Y1^2 + 2*Y1*Y2 - Y2^2 = X1*(X1-X2)^2 - (Y1-Y2)^2 

f6= X1 - X2 - 1
f7 = X1 - X2 - 2

f8 = Y1*Y2 - 3*b
f9 = Y1*Y2 + 3*b
f10 = X2*Y1 + X1*Y2
f11 = X1*X2 + Y1*Y2"""

'jacobian-0:madd, jacobian-0:add-1998-cmo, jacobian-0:add-2007-bl, jacobian-0:add-1998-cmo-2, jacobian-0:mmadd-2007-bl, jacobian-0:madd-2007-bl, jacobian-0:madd-2008-g, projective:add-1998-cmo, projective:add-1998-cmo-2, projective:madd-1998-cmo, projective:mmadd-1998-cmo, jacobian:madd, jacobian:add-1998-cmo, jacobian:add-2007-bl, jacobian:add-1998-cmo-2, jacobian:mmadd-2007-bl, jacobian:madd-2007-bl, jacobian:madd-2008-g, xyzz:add-2008-s, xyzz:madd-2008-s, xyzz:mmadd-2008-s, modified:madd-2009-bl, modified:add-1998-cmo-2, modified:add-2009-bl, (2): \n{f4} \n\njacobian-0:zadd-2007-m, jacobian:zadd-2007-m, (2): \n{f5} \n\njacobian-0:madd-2004-hmv, jacobian:madd-2004-hmv, (2): \n{f2} \n\njacobian-0:add-1986-cc, jacobian-0:add-2001-b, jacobian:add-1986-cc, jacobian:add-2001-b, (2): \n{f1} \n\nprojective:add-2002-bj, projective:add-2007-bl, (2): \n{f1, f3} \n\nmodified:mmadd-2009-bl, (4): \n{f4, f6, f7} \n\n\nprojective:madd-2015-rcb (5):\n{f1, f8, f9, f10, f11}\n\n\nf1 = X1 + X2\nf2 = 2*